# Modul 05 · nb07 — Capstone: Ask-My-Document

Notebook ini **menyatukan semua** yang sudah kita pelajari menjadi satu produk: tanya-jawab atas dokumen PDF apa pun, dengan **jawaban bersitasi** — setiap klaim menunjuk halaman + bagian sumbernya, dan sitasinya **diverifikasi**.

| Tahap | Dari notebook | Teknik |
|------|---------------|--------|
| 1. Ingest PDF | nb02 | Docling parse (struktur, tabel) |
| 2. Chunk + metadata | nb02 | HybridChunker sadar-struktur + halaman/heading |
| 3. Retrieve + rerank | nb03 | FAISS bi-encoder → cross-encoder bge-reranker-v2-m3 |
| 4. Percakapan | nb06 | memori window+summary + history-aware rewriting |
| 5. **Jawaban bersitasi + verifikasi** | **baru** | label sumber `[S#]`, dicek dengan `cited_labels` |

```
PDF (URL/upload) → Docling → chunk (+halaman/heading)
        → embed → FAISS → rerank (bge-reranker-v2-m3)
        → top-k chunk berlabel [S1]..[S4]
        → LLM menjawab + menyebut [S#]  → verifikasi label  → tampilkan jawaban + sumber
```

> **Dokumen contoh:** default mengambil paper publik *"Attention Is All You Need"* dari arXiv — **langsung jalan tanpa upload**. Ganti `DOC_URL` ke PDF publik lain, atau set `UPLOAD=True` untuk PDF sendiri.
>
> **Generator:** default **NVIDIA NIM** (cloud, Llama-3.3-70B — andal, butuh `NVIDIA_API_KEY` di Colab Secrets). Set `GENERATOR="qwen"` untuk **Qwen2.5-3B 4-bit** lokal (butuh **GPU T4**). Reranker + embedder tetap lokal (lebih cepat dengan T4).

In [ ]:
# Install dependencies
!pip install -q docling "transformers<5" "sentence-transformers>=3.0" faiss-cpu accelerate bitsandbytes openai

import os, sys

# Clone the course repo so we can import the tested helpers + use the sample PDF fallback
REPO = "/content/navasena-gen-ml-course"
if not os.path.exists(REPO):
    !git clone https://github.com/chmdznr/navasena-gen-ml-course.git {REPO}
sys.path.append(os.path.abspath(f"{REPO}/05_rag"))

from tools.rag_utils import (ConversationalMemoryManager, format_pages, source_label, cited_labels)
print("Helpers OK ->", source_label(1, [6, 7], "Contoh Heading"))

## Langkah 1 — Ingest: ambil & parse PDF

Docling mengubah PDF (termasuk struktur heading & tabel) menjadi `DoclingDocument`. Sumbernya bisa **URL** (diunduh otomatis) atau **berkas lokal** hasil upload.

- **Default (tanpa upload):** `DOC_URL` menunjuk paper publik di arXiv.
- **Pakai dokumenmu sendiri:** set `UPLOAD = True`, lalu pilih file PDF saat diminta.
- **Jaring pengaman:** bila URL gagal diakses, notebook memakai contoh PDF di repo (Candi Borobudur) — ditandai dengan banner agar kamu tahu dokumennya berganti.

In [ ]:
# ── PILIH SUMBER DOKUMEN ───────────────────────────────────────────────────
DOC_URL = "https://arxiv.org/pdf/1706.03762"      # "Attention Is All You Need"
UPLOAD  = False                                   # True -> upload PDF sendiri dari komputer

from docling.document_converter import DocumentConverter

if UPLOAD:
    from google.colab import files
    uploaded = files.upload()                     # pilih file di dialog; tersimpan di /content
    source = next(iter(uploaded))                 # nama berkasnya jadi path lokal
else:
    source = DOC_URL

print(f"Memproses: {source}")
converter = DocumentConverter()
used_fallback = False
try:
    doc = converter.convert(source).document
except Exception as e:
    used_fallback = True
    source = f"{REPO}/05_rag/data/sample_id_document.pdf"
    print("=" * 64)
    print(f"⚠️  GAGAL ambil dokumen ({type(e).__name__}).")
    print(f"    Memakai contoh repo: {source} (isi: Candi Borobudur).")
    print("    Pertanyaan demo bawaan (tentang paper Transformer) tidak akan relevan —")
    print("    ganti daftar `pertanyaan` di sel demo sesuai isi dokumen contoh.")
    print("=" * 64)
    doc = converter.convert(source).document

n_pages = len(getattr(doc, "pages", {}) or {})
print(f"Selesai. Halaman terdeteksi: {n_pages}")

## Langkah 2 — Chunk + metadata untuk sitasi

`HybridChunker` memotong dokumen **mengikuti struktur** (bagian/heading), bukan sekadar per-N-kata. Untuk tiap chunk kita simpan **nomor halaman** dan **heading** — bahan baku sitasi.

Dua hal penting:
- **`contextualize(chunk)`** menambahkan jejak heading di depan teks; kita **meng-embed versi ini** (bukan teks polos) agar retrieval lebih akurat.
- Tokenizer chunker = tokenizer model embedding, supaya tiap chunk muat dalam 512 token.

In [ ]:
# ── Chunk sadar-struktur + metadata sitasi (halaman & heading per chunk) ──
from docling.chunking import HybridChunker
EMB_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def build_chunker():
    # Tokenizer chunker = model embedding (chunk <=512 token). Coba API baru -> string -> default.
    try:
        from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
        from transformers import AutoTokenizer
        ch = HybridChunker(tokenizer=HuggingFaceTokenizer(
            tokenizer=AutoTokenizer.from_pretrained(EMB_NAME), max_tokens=512))
        print("Chunker: tokenizer embedding (512 token) — sesuai resep.")
        return ch
    except Exception:
        try:
            ch = HybridChunker(tokenizer=EMB_NAME, max_tokens=512)
            print("Chunker: tokenizer string (512 token).")
            return ch
        except Exception:
            print("⚠️  Chunker: tokenizer DEFAULT — budget 512 token tidak dijamin, retrieval bisa menurun.")
            return HybridChunker()

chunker = build_chunker()
dchunks = list(chunker.chunk(dl_doc=doc))

def pages_heading(ch):
    """(daftar halaman, heading terakhir) dari satu chunk Docling -> bahan sitasi."""
    pages, heads = set(), []
    meta = getattr(ch, "meta", None)
    if meta is not None:
        heads = list(getattr(meta, "headings", []) or [])
        for item in (getattr(meta, "doc_items", []) or []):
            for prov in (getattr(item, "prov", []) or []):
                if getattr(prov, "page_no", None) is not None:
                    pages.add(prov.page_no)
    return sorted(pages), (heads[-1] if heads else None)

texts       = [c.text for c in dchunks]                       # ditampilkan ke pengguna
embed_texts = [chunker.contextualize(c) for c in dchunks]     # PREPEND heading -> EMBED ini (lebih akurat)
metas       = [pages_heading(c) for c in dchunks]             # (pages, heading) tiap chunk

if dchunks:
    print(f"{len(dchunks)} chunk dibuat.")
    _p, _h = metas[min(2, len(metas) - 1)]
    print("Contoh metadata sitasi ->", source_label(1, _p, _h))
else:
    print("⚠️  0 chunk — dokumen mungkin kosong atau hasil scan tanpa teks.")

## Langkah 3 — Index, pencarian dua tahap, & generator

**Retrieval** (sama seperti nb03): **bi-encoder** ambil 20 kandidat (cepat) → **cross-encoder** `bge-reranker-v2-m3` urutkan ulang, ambil 4 teratas (akurat). Reranker inilah yang melakukan seleksi lintas-bahasa paling tajam.

**Generator** (seperti nb06): default **NVIDIA NIM** (Llama-3.3-70B, cloud, andal) — set `GENERATOR="qwen"` untuk Qwen2.5-3B 4-bit lokal. Generatornya hanya perlu mengikuti instruksi sitasi, jadi kualitas jawaban naik seiring kapasitas model.

In [ ]:
# ── Embedder + FAISS + reranker (retrieval dua tahap) ──
import faiss, numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

embedder = SentenceTransformer(EMB_NAME)
embedder.max_seq_length = 512                       # samakan dgn budget chunker (default 128 -> truncate ekor chunk)
cvecs = embedder.encode(embed_texts, convert_to_numpy=True, show_progress_bar=True).astype("float32")
index = faiss.IndexFlatL2(cvecs.shape[1]); index.add(cvecs)

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

def retrieve_rerank(query, k_over=20, k_top=4):
    """Stage 1: bi-encoder over-fetch (FAISS). Stage 2: cross-encoder rerank -> top_k ids."""
    if not texts:
        return []
    qv = embedder.encode([query], convert_to_numpy=True).astype("float32")
    _, idx = index.search(qv, min(k_over, len(texts))); cand = idx[0].tolist()
    scores = reranker.predict([(query, texts[c]) for c in cand]).tolist()
    order = sorted(range(len(cand)), key=lambda i: scores[i], reverse=True)[:k_top]
    return [cand[i] for i in order]

print("Embedder + index + reranker siap.")

In [ ]:
# ── Pilih generator: "nim" (cloud NVIDIA NIM, andal, butuh API key) atau "qwen" (lokal, butuh GPU T4) ──
GENERATOR = "nim"     # ganti ke "qwen" untuk memuat Qwen2.5-3B 4-bit secara lokal

if GENERATOR == "nim":
    import os, getpass
    from openai import OpenAI                       # NVIDIA NIM kompatibel dengan API OpenAI
    try:
        from google.colab import userdata
        os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY") or ""
    except Exception:
        if not os.environ.get("NVIDIA_API_KEY"):
            os.environ["NVIDIA_API_KEY"] = getpass.getpass("NVIDIA_API_KEY: ")
    assert os.environ.get("NVIDIA_API_KEY"), "GENERATOR='nim' butuh NVIDIA_API_KEY (simpan di Colab Secrets)."
    _client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=os.environ["NVIDIA_API_KEY"])
    NIM_MODEL = "meta/llama-3.3-70b-instruct"
    def generate(prompt, max_new_tokens=320):
        r = _client.chat.completions.create(
            model=NIM_MODEL, temperature=0, max_tokens=max_new_tokens,
            messages=[{"role": "system", "content": "Jawab dalam Bahasa Indonesia, ringkas dan akurat."},
                      {"role": "user", "content": prompt}])
        return r.choices[0].message.content.strip()
    print("Generator: NVIDIA NIM ->", NIM_MODEL)
else:
    import warnings, torch
    warnings.filterwarnings("ignore", message=".*_check_is_size.*")   # bitsandbytes internal, non-fatal
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    _name = "Qwen/Qwen2.5-3B-Instruct"
    tok = AutoTokenizer.from_pretrained(_name)
    _model = AutoModelForCausalLM.from_pretrained(_name, quantization_config=bnb, device_map="auto")
    for _k in ("temperature", "top_p", "top_k"):     # do_sample=False -> buang flag sampling
        setattr(_model.generation_config, _k, None)
    def generate(prompt, max_new_tokens=320):
        msgs = [{"role": "system", "content": "Jawab dalam Bahasa Indonesia, ringkas dan akurat."},
                {"role": "user", "content": prompt}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = tok(text, return_tensors="pt").to(_model.device)
        out = _model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
        return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print("Generator: Qwen2.5-3B lokal ->", _model.device)

## Langkah 4 — Percakapan + jawaban bersitasi & verifikasi (inti capstone)

Tiap pertanyaan: **history-aware rewriting** (nb06) → retrieve+rerank → tiap chunk teratas diberi label `[S1]..[S4]` beserta halaman/heading → LLM menjawab **hanya** dari konteks dan **menyebut `[S#]`**. Lalu `cited_labels` **memverifikasi** label yang ditulis model: yang valid ditandai ✓, yang di luar daftar ditandai ⚠️.

> Verifikasi ini penting: model bisa salah mengarang label. Dengan menampilkan **sumber asli** (halaman + heading) dan menandai sitasi yang valid, pembaca bisa menelusuri tiap klaim ke dokumen — apa pun generatornya.

In [ ]:
# ── Memori + rewriting + jawaban bersitasi + verifikasi sitasi ──
def summarize_turn(old_summary, turn):
    u, a = turn
    return generate(f"Ringkasan saat ini: {old_summary or '(kosong)'}\n"
                    f"Tambahkan giliran ini lalu ringkas ulang singkat -> User: {u}\nAsisten: {a}\n"
                    f"Ringkasan baru (1-2 kalimat):", 80)

def rewrite_followup(history_context, follow_up):
    if not history_context.strip():
        return follow_up
    return generate(f"Riwayat percakapan:\n{history_context}\n\nPertanyaan lanjutan: {follow_up}\n\n"
                    f"Tulis ulang jadi SATU pertanyaan mandiri singkat. PERTAHANKAN topik pertanyaan "
                    f"lanjutan; pakai riwayat hanya untuk mengisi kata ganti/predikat yang hilang. "
                    f"Keluarkan HANYA pertanyaannya:", 60)

memory = ConversationalMemoryManager(summarize_fn=summarize_turn, window=3)

def ask(question):
    """Satu giliran: rewrite -> retrieve+rerank -> jawab dengan [S#] -> verifikasi label."""
    standalone = rewrite_followup(memory.context(), question)
    ids = retrieve_rerank(standalone)
    blocks, sources = [], []
    for n, cid in enumerate(ids, 1):
        pages, heading = metas[cid]
        blocks.append(f"[S{n}] (hlm {format_pages(pages)}) {texts[cid]}")   # teks penuh chunk -> grounding kuat
        sources.append(source_label(n, pages, heading))
    context = "\n\n".join(blocks)
    answer = generate(f"Konteks (tiap sumber berlabel [S#]):\n{context}\n\nPertanyaan: {standalone}\n\n"
                      f"Jawab HANYA berdasarkan konteks dan bubuhkan label [S#] pada klaim yang relevan. "
                      f"Bila konteks tidak memuat jawabannya, katakan informasinya tidak tersedia.")
    memory.add_turn(question, answer)
    valid, invalid = cited_labels(answer, len(ids))
    return standalone, answer, sources, valid, invalid

def show(question):
    standalone, answer, sources, valid, invalid = ask(question)
    print(f"❓ {question}")
    if standalone != question:
        print(f"   (ditulis ulang -> {standalone})")
    print(f"💬 {answer}\n📚 Lihat sumber:")
    for n, s in enumerate(sources, 1):
        print(f"   {s}{'   ✓ dikutip' if n in valid else ''}")
    if invalid:
        bad = ", ".join(f"[S{i}]" for i in invalid)
        print(f"   ⚠️  jawaban menyebut {bad} di luar daftar sumber — sitasi tak valid, periksa!")

print("Pipeline Ask-My-Document siap.")

## Demo — bertanya dalam Bahasa Indonesia tentang paper berbahasa Inggris

Pertanyaan Bahasa Indonesia tetap menemukan bagian relevan di dokumen berbahasa Inggris: embedding-nya **multilingual**, dan **reranker** `bge-reranker-v2-m3` (juga multilingual) melakukan seleksi akhir lintas-bahasa. Perhatikan giliran ke-2 ("...mengatasi masalah **itu**?") — kata ganti "itu" diselesaikan history-aware rewriting, dan tiap jawaban menandai sumber yang **terverifikasi** (✓).

In [ ]:
# ── Demo: tanya dokumen dalam Bahasa Indonesia (lintas-bahasa + sitasi terverifikasi) ──
memory.clear()
print(f"📄 Dokumen aktif: {source}  ({n_pages} halaman, {len(texts)} chunk)\n")

# Pertanyaan ini mengasumsikan dokumen DEFAULT (paper "Attention Is All You Need").
# Kalau kamu mengganti DOC_URL / upload dokumen lain / memakai fallback, sesuaikan daftar ini.
pertanyaan = [
    "Apa masalah utama arsitektur lama (RNN) yang ingin diatasi paper ini?",
    "Bagaimana self-attention mengatasi masalah itu?",      # 'itu' -> menguji history-aware rewriting
    "Berapa banyak attention head yang dipakai di model dasar?",
]
for q in pertanyaan:
    show(q)
    print("─" * 70)

## Cobai sendiri — Q&A interaktif

Jalankan sel di bawah, ketik pertanyaan tentang isi dokumen, lalu **tekan Enter**. Saat menjawab muncul indikator `⏳ mencari + menjawab` (NIM/Qwen butuh beberapa detik). Perintah: `stats` (memori), `clear` (kosongkan memori), `quit` (berhenti).

In [ ]:
# ── Ngobrol interaktif dengan dokumenmu. Ketik lalu TEKAN ENTER. ──
# Perintah: 'quit' (berhenti), 'stats' (memori), 'clear' (kosongkan memori).
memory.clear()
print("=== Ask-My-Document ===")
print(f"📄 Dokumen aktif: {source}")
print("Tanya apa saja tentang isi dokumen. Ketik 'quit' untuk berhenti.\n")
try:
    while True:
        q = input("Tanya: ").strip()
        if not q:
            continue
        if q.lower() == "quit":
            print("Selesai. Sampai jumpa!"); break
        if q.lower() == "stats":
            print("  ", memory.stats()); continue
        if q.lower() == "clear":
            memory.clear(); print("  (memori dikosongkan)"); continue
        print("  ⏳ mencari + menjawab (~beberapa detik)...")
        show(q)
        print()
except (KeyboardInterrupt, EOFError):
    print("\nDihentikan.")

## Ringkasan & jembatan ke nb08

### Yang kita bangun
Satu pipeline **Ask-My-Document** utuh: PDF (URL/upload) → Docling → chunk sadar-struktur (+halaman/heading) → FAISS → rerank → **jawaban bersitasi + verifikasi** + percakapan multi-turn, dengan generator yang bisa ditukar (NIM cloud / Qwen lokal).

| Komponen | Asal |
|----------|------|
| Ingest PDF (URL/upload) | nb02 (Docling) |
| Chunk + metadata halaman/heading | nb02 + `format_pages` / `source_label` (rag_utils) |
| Retrieve + rerank | nb03 (bi → cross-encoder) |
| Memori + rewriting | nb06 (`ConversationalMemoryManager`) |
| Jawaban bersitasi + verifikasi `[S#]` | nb07 (`cited_labels`) |

### Jembatan ke nb08 — Deploy
nb08 membungkus pipeline ini menjadi layanan: endpoint `/ask` (FastAPI) yang mengembalikan **jawaban + sumber** dalam JSON, generator **NVIDIA NIM** hosted, dan runbook agar bisa dijalankan di mesin sendiri / edge.